# HumToSing: AudioCraft quality trial
Personal, noncommercial research only. Use **a free Colab GPU runtime** (Runtime → Change runtime type → GPU). No purchases, credits, API subscription, web server or background service are needed. Free GPU availability is not guaranteed; if unavailable, stop and try another time.

This is an interactive experiment, not the app's production backend. Only the files you select in the upload cell leave your Mac. Do not upload whole projects or private lyrics. Do not mount Google Drive. Original hum, generated audio and transcribed MIDI stay separate.

Run the cells in order. Installation/model downloads may take several minutes. Model and dependency installation are not yet validated on your allocated Colab runtime. Any installation or memory error stops the trial; do not switch to a text-only model.

After setup, run **Upload → Generate → Download** once. Select both trial files; generation starts after validation and the result downloads automatically. Progress and timings appear here in Colab. Do not import guide.wav into the app as a generated result.


In [ ]:
# Check the allocated GPU before installing or accepting recordings.
import subprocess, sys, pathlib, json
try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True).strip().splitlines()[0]
    memory_mib = int(gpu.rsplit(",", 1)[1].strip())
    assert memory_mib >= 14 * 1024, "A GPU with roughly 16 GB memory is needed."
except Exception as error:
    raise RuntimeError("No suitable free GPU. Stop; no paid upgrade or CPU fallback. " + str(error))
print("Allocated GPU:", gpu)


In [ ]:
# Isolate the old PyTorch stack from Colab's own Python/packages.
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.6.17"], check=True)
UV = [sys.executable, "-m", "uv"]
ENV = pathlib.Path("/content/hum-audiocraft-env")
if ENV.exists():
    raise RuntimeError("Environment already exists. Continue with the next cell, or delete the runtime to start fresh.")
subprocess.run(UV + ["venv", "--python", "3.10", "--seed", str(ENV)], check=True)
PYTHON = str(ENV / "bin/python")
# PyAV may need the distribution's FFmpeg headers when no binary wheel is available.
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "pkg-config", "libavformat-dev", "libavcodec-dev", "libavdevice-dev", "libavutil-dev", "libswscale-dev", "libswresample-dev", "libavfilter-dev"], check=True)


In [ ]:
CONSTRAINTS = '# Compatibility constraints for the pinned AudioCraft source (Python 3.10).\nnumpy==1.26.4\ntorch==2.1.0\ntorchaudio==2.1.0\ntorchvision==0.16.0\ntorchtext==0.16.0\nxformers==0.0.22.post7\ntransformers==4.35.2\nhuggingface-hub==0.25.2\ngradio==3.50.2\ntorchmetrics==1.3.2\nspacy==3.7.6\nav==11.0.0\nsetuptools==69.5.1\n'
pathlib.Path("/content/hum-constraints.txt").write_text(CONSTRAINTS)
PIP = [PYTHON, "-m", "pip"]
subprocess.run(PIP + ["install", "pip==24.3.1", "setuptools==69.5.1", "wheel==0.45.1"], check=True)
subprocess.run(PIP + ["install", "-c", "/content/hum-constraints.txt", "torch==2.1.0", "torchaudio==2.1.0", "torchvision==0.16.0"], check=True)
subprocess.run(PIP + ["install", "-c", "/content/hum-constraints.txt", "git+https://github.com/facebookresearch/audiocraft.git@896ec7c47f5e5d1e5aa1e4b260c4405328bf009d"], check=True)
subprocess.run(PIP + ["check"], check=True)
subprocess.run([PYTHON, "-c", "import torch, audiocraft; print('Torch', torch.__version__); assert torch.cuda.is_available(), 'CUDA unavailable'"], check=True)


In [ ]:
RUNNER_SOURCE = '"""Interactive MusicGen Melody trial. Imports ML libraries only after validation.\nNo server, upload client, fallback model, automatic retry or billing operation.\n"""\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport time\nimport wave\n\nMODEL = "facebook/musicgen-melody"\nINSTRUMENTS = ("piano", "acoustic-guitar", "flute")\nSTYLES = ("neutral", "gentle", "bright", "melancholic", "energetic")\nAUDIOCRAFT_REVISION = "896ec7c47f5e5d1e5aa1e4b260c4405328bf009d"\n\n\ndef validate_request(request_path, guide_path):\n    request_path, guide_path = Path(request_path), Path(guide_path)\n    if request_path.stat().st_size > 8192 or guide_path.stat().st_size > 4_000_044:\n        raise ValueError("Oversized trial input")\n    request = json.loads(request_path.read_text(encoding="utf-8"))\n    if not isinstance(request, dict):\n        raise ValueError("Request must be an object")\n    if type(request.get("schema_version")) is not int or request["schema_version"] != 1:\n        raise ValueError("Unsupported request version")\n    if request.get("model") != MODEL or request.get("audio_file") != "guide.wav":\n        raise ValueError("Only MusicGen Melody with guide.wav is permitted")\n    if request.get("instrument") not in INSTRUMENTS or request.get("style") not in STYLES:\n        raise ValueError("Unknown instrument or style")\n    for key, low, high in (("tempo_bpm", 20, 400), ("duration_seconds", .5, 10)):\n        value = request.get(key)\n        if type(value) not in (int, float) or not math.isfinite(value) or not low <= value <= high:\n            raise ValueError("Invalid " + key)\n    seed = request.get("seed")\n    if type(seed) is not int or not 0 <= seed <= 0xFFFFFFFF:\n        raise ValueError("Seed must be a 32-bit unsigned integer")\n    prompt = request.get("prompt")\n    if not isinstance(prompt, str) or not prompt.strip() or len(prompt) > 1000:\n        raise ValueError("Missing or oversized prompt")\n    with wave.open(str(guide_path), "rb") as wav:\n        rate, frames = wav.getframerate(), wav.getnframes()\n        if wav.getnchannels() != 1 or wav.getsampwidth() != 2 or not 8000 <= rate <= 192000:\n            raise ValueError("Guide must be mono PCM16 WAV at 8–192 kHz")\n        if not .5 <= frames / rate <= 10 or abs(frames / rate - request["duration_seconds"]) > 1 / rate:\n            raise ValueError("Guide duration does not match request")\n        pcm = wav.readframes(frames)\n        if len(pcm) != frames * 2:\n            raise ValueError("Truncated guide audio")\n        import array\n        import sys\n        samples = array.array("h", pcm)\n        if sys.byteorder != "little":\n            samples.byteswap()\n        if max(abs(x) for x in samples) <= 32:\n            raise ValueError("Guide is effectively silent")\n    return request\n\n\ndef gpu_preflight(torch):\n    if not torch.cuda.is_available():\n        raise RuntimeError("No CUDA GPU. Stop and request a free GPU runtime; no CPU/text-only fallback.")\n    props = torch.cuda.get_device_properties(0)\n    # A T4 reports ~15 GiB usable memory despite being marketed as 16 GB.\n    if props.total_memory < 14 * 1024 ** 3:\n        raise RuntimeError("Insufficient GPU memory for this trial. No paid upgrade will be attempted.")\n    return {"name": props.name, "total_memory_bytes": props.total_memory}\n\n\ndef run(request_path, guide_path, output_dir, progress=None):\n    report = progress or (lambda stage, completed, total: None)\n    request = validate_request(request_path, guide_path)\n    output = Path(output_dir)\n    if output.exists():\n        raise ValueError("Use a new output directory; previous results are never overwritten")\n    report("loading", 0, 0)\n    import torch\n    gpu = gpu_preflight(torch)\n    import torchaudio\n    from huggingface_hub import HfApi, snapshot_download\n    from audiocraft.models import MusicGen\n\n    # Resolve once, pin this run, and record the exact model snapshot used.\n    revision = HfApi().model_info(MODEL).sha\n    if not isinstance(revision, str) or len(revision) != 40 or any(c not in "0123456789abcdef" for c in revision):\n        raise RuntimeError("Could not resolve an immutable model revision")\n    snapshot = snapshot_download(MODEL, revision=revision,\n                                 allow_patterns=["state_dict.bin", "compression_state_dict.bin"])\n    load_started = time.monotonic()\n    model = MusicGen.get_pretrained(snapshot, device="cuda")\n    model.set_custom_progress_callback(lambda completed, total: report("generating", completed, total))\n    # Seed after initialization so loading the model does not consume the sampling seed.\n    torch.manual_seed(request["seed"])\n    torch.cuda.manual_seed_all(request["seed"])\n    model.set_generation_params(duration=request["duration_seconds"], use_sampling=True,\n                                top_k=250, top_p=0.0, temperature=1.0, cfg_coef=3.0)\n    melody, rate = torchaudio.load(str(guide_path))\n    load_seconds = time.monotonic() - load_started\n    torch.cuda.synchronize()\n    started = time.monotonic()\n    report("generating", 0, 0)\n    with torch.inference_mode():\n        result = model.generate_with_chroma([request["prompt"]], melody.unsqueeze(0), rate,\n                                            progress=True)\n    torch.cuda.synchronize()\n    elapsed = time.monotonic() - started\n    result = result[0].detach().cpu()\n    if result.ndim != 2 or result.shape[0] != 1 or not torch.isfinite(result).all():\n        raise RuntimeError("Unexpected or non-finite model output")\n    expected = round(request["duration_seconds"] * model.sample_rate)\n    if result.shape[-1] < expected - model.sample_rate // 10:\n        raise RuntimeError("Generated output is unexpectedly short")\n    result = result[:, :expected]\n    peak = result.abs().max().item()\n    if peak <= 1e-5:\n        raise RuntimeError("Model returned silence; no successful result recorded")\n    gain = min(1.0, .95 / peak)\n    result *= gain\n    packages = {dist.metadata["Name"]: dist.version for dist in importlib.metadata.distributions() if dist.metadata["Name"]}\n    actual_direct = importlib.metadata.distribution("audiocraft").read_text("direct_url.json")\n    metadata = dict(request)\n    metadata.update(model_revision=revision, audiocraft_revision=AUDIOCRAFT_REVISION,\n                    audiocraft_install=json.loads(actual_direct) if actual_direct else None,\n                    guide_sha256=hashlib.sha256(Path(guide_path).read_bytes()).hexdigest(),\n                    generation_seconds=elapsed, load_seconds=load_seconds, output_gain=gain,\n                    output_sample_rate=model.sample_rate, output_frames=result.shape[-1],\n                    gpu=gpu, quality="unrated", sampling={"top_k": 250, "temperature": 1.0, "cfg_coef": 3.0})\n    report("saving", 0, 0)\n    output.mkdir(parents=True, exist_ok=False)\n    try:\n        torchaudio.save(str(output / "generated.wav"), result, model.sample_rate,\n                       encoding="PCM_S", bits_per_sample=16)\n        (output / "result.json").write_text(json.dumps(metadata, indent=2) + "\\n", encoding="utf-8")\n        (output / "environment.json").write_text(json.dumps({"python": platform.python_version(), "packages": packages}, indent=2) + "\\n")\n        (output / "SUCCESS").write_text("Generation completed; listening and MIDI evaluation still required.\\n")\n    except Exception:\n        # Keep any partial output marked incomplete; never publish it as successful.\n        (output / "INCOMPLETE").write_text("Output writing failed. Use a fresh result directory.\\n")\n        raise\n    report("complete", 1, 1)\n    return metadata\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--request", required=True)\n    parser.add_argument("--guide", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--progress-json", action="store_true")\n    args = parser.parse_args()\n    try:\n        def progress(stage, completed, total):\n            print("HUM_PROGRESS " + json.dumps(dict(stage=stage, completed=completed, total=total)), flush=True)\n        run(args.request, args.guide, args.output, progress if args.progress_json else None)\n    except Exception as error:\n        raise SystemExit("Trial stopped (no automatic retry or paid fallback): " + str(error))\n'
RUNNER = pathlib.Path("/content/hum_trial_runner.py")
RUNNER.write_text(RUNNER_SOURCE)


In [ ]:
WORKFLOW_SOURCE = '"""Interactive Colab helpers; no HTTP service or desktop upload endpoint."""\nimport json\nfrom pathlib import Path\nimport queue\nimport subprocess\nimport threading\nimport time\nimport uuid\n\n\ndef run_generation(python, runner, request, guide, output, on_progress):\n    """Forward actual model steps, keep elapsed time live, and stop on interruption."""\n    started = time.monotonic()\n    process = subprocess.Popen([str(python), \'-u\', str(runner), \'--request\', str(request),\n                                \'--guide\', str(guide), \'--output\', str(output), \'--progress-json\'],\n                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n    lines = queue.Queue()\n    def read():\n        try:\n            for line in process.stdout:\n                lines.put(line)\n        finally:\n            lines.put(None)\n    reader = threading.Thread(target=read, daemon=True)\n    reader.start()\n    state = dict(stage=\'loading\', completed=0, total=0)\n    tail = []\n    try:\n        while True:\n            try:\n                line = lines.get(timeout=.5)\n            except queue.Empty:\n                line = \'\'\n            if line is None:\n                break\n            if line.startswith(\'HUM_PROGRESS \'):\n                event = json.loads(line[len(\'HUM_PROGRESS \'):])\n                if (event.get(\'stage\') not in (\'loading\', \'generating\', \'saving\', \'complete\')\n                        or type(event.get(\'completed\')) is not int or type(event.get(\'total\')) is not int\n                        or not 0 <= event[\'completed\'] <= event[\'total\']):\n                    raise ValueError(\'Invalid model progress event\')\n                state = event\n            elif line:\n                tail.append(line.rstrip()[-2000:])\n                tail = tail[-12:]\n            on_progress(dict(state, elapsed=time.monotonic() - started))\n        code = process.wait()\n        if code != 0 or not (Path(output) / \'SUCCESS\').is_file():\n            raise RuntimeError(\'Generation stopped; no result will be downloaded.\\n\' + \'\\n\'.join(tail))\n        on_progress(dict(stage=\'complete\', completed=1, total=1, elapsed=time.monotonic() - started))\n    finally:\n        if process.poll() is None:\n            process.terminate()\n            try:\n                process.wait(timeout=5)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait()\n        reader.join(timeout=5)\n        process.stdout.close()\n\n\n# Transfer result bytes over the active notebook kernel channel. No tunnel/server.\n# Completion means bytes reached the browser; the user\'s disk-save dialog is separate.\nDOWNLOAD_JS = r\'\'\'async (channelId, filename, size) => {\n  const box = document.createElement(\'div\');\n  const title = document.createElement(\'div\');\n  const meter = document.createElement(\'progress\');\n  meter.max = size; meter.value = 0; meter.style.width = \'100%\';\n  box.append(title, meter); document.body.append(box);\n  const began = performance.now();\n  let received = 0;\n  const update = () => {\n    const seconds = (performance.now() - began) / 1000;\n    const eta = received > 0 ? seconds * (size - received) / received : null;\n    title.textContent = `Downloading result: ${Math.floor(100 * received / size)}% · ${seconds.toFixed(1)} s elapsed` +\n      (eta === null ? \'\' : ` · ~${eta.toFixed(1)} s remaining`);\n  };\n  const clock = setInterval(update, 100);\n  let channel;\n  try {\n    if (!google.colab.kernel.accessAllowed) throw new Error(\'Notebook access is not allowed\');\n    channel = await google.colab.kernel.comms.open(channelId);\n    const pieces = [];\n    channel.send({});\n    for await (const packet of channel.messages) {\n      for (const piece of packet.buffers || []) {\n        received += piece.byteLength;\n        if (received > size) throw new Error(\'Unexpected download size\');\n        pieces.push(piece);\n      }\n      meter.value = received; update(); channel.send({});\n    }\n    if (received !== size) throw new Error(\'Download interrupted; rerun the download cell\');\n    const url = URL.createObjectURL(new Blob(pieces, {type: \'application/zip\'}));\n    const link = document.createElement(\'a\');\n    link.href = url; link.download = filename; link.textContent = \'Save result ZIP again\';\n    box.append(link); link.click();\n    setTimeout(() => URL.revokeObjectURL(url), 300000);\n    const seconds = (performance.now() - began) / 1000;\n    title.textContent = `Result transferred to browser in ${seconds.toFixed(1)} s. Check Downloads or the save dialog.`;\n    return {bytes: received, seconds};\n  } catch (error) {\n    title.textContent = `Download failed: ${error.message}. Rerun Download again below; do not regenerate.`;\n    throw error;\n  } finally {\n    clearInterval(clock);\n    if (channel && typeof channel.close === \'function\') channel.close();\n  }\n}\'\'\'\n\n\ndef download_result(path):\n    from google.colab import output\n    from IPython import get_ipython\n    path = Path(path)\n    size = path.stat().st_size\n    if not 0 < size <= 32 * 1024 * 1024:\n        raise ValueError(\'Invalid result archive size\')\n    manager = get_ipython().kernel.comm_manager\n    key = \'hum_result_\' + uuid.uuid4().hex\n    streams = []\n    def opened(comm, message):\n        stream = path.open(\'rb\')\n        streams.append(stream)\n        def next_chunk(message):\n            block = stream.read(256 * 1024)\n            if block:\n                comm.send({}, buffers=[block])\n            else:\n                stream.close()\n                comm.close()\n        comm.on_msg(next_chunk)\n        comm.on_close(lambda message: stream.close())\n    manager.register_target(key, opened)\n    try:\n        return output.eval_js(\'(\' + DOWNLOAD_JS + \')(\' + \',\'.join(map(json.dumps, (key, path.name, size))) + \')\', timeout_sec=600)\n    finally:\n        manager.unregister_target(key, opened)\n        for stream in streams:\n            stream.close()\n'
WORKFLOW = pathlib.Path("/content/hum_notebook_workflow.py")
WORKFLOW.write_text(WORKFLOW_SOURCE)


## Upload → Generate → Download
Run the next cell using its ▶ button, then select **guide.wav and request.json together**. Upload progress appears in the chooser output; model loading is indeterminate, generation uses real model-step progress, and downloading shows bytes, elapsed time and an estimated remaining time. This cell generates exactly one result with the displayed request. Keep the notebook open. Use Stop execution to interrupt generation.


In [ ]:
from google.colab import files
from IPython.display import HTML, Audio, display
import importlib.util, time, uuid, zipfile

archive = None  # A failed new trial must not offer a previous result.

def load_helper(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

runner = load_helper("hum_trial_runner", RUNNER)
workflow = load_helper("hum_notebook_workflow", WORKFLOW)
JOB = pathlib.Path("/content/hum-trials") / uuid.uuid4().hex
INPUT = JOB / "input"
INPUT.mkdir(parents=True, exist_ok=False)
print("Select guide.wav AND request.json from the same saved trial folder. Generation starts after upload and validation.")
upload_started = time.monotonic()
uploaded = files.upload(target_dir=str(INPUT))
if {pathlib.Path(name).name for name in uploaded} != {"request.json", "guide.wav"} or len(uploaded) != 2:
    raise ValueError("Select exactly guide.wav and request.json. Rerun this cell to retry; nothing was generated.")
print(f"Upload finished in {time.monotonic() - upload_started:.1f} seconds (including file selection).")
REQUEST = runner.validate_request(INPUT / "request.json", INPUT / "guide.wav")
print("Generating:", REQUEST["prompt"])
OUTPUT = JOB / "result"
progress_display = display(HTML("Preparing generation…"), display_id=True)
def show_progress(event):
    labels = {"loading": "Loading model (first use can take several minutes)", "generating": "Generating instrument", "saving": "Saving generated audio", "complete": "Generation complete"}
    total = event["total"]
    percent = 100 * event["completed"] / total if total else None
    meter = '<progress style="width:100%"></progress>' if percent is None else f'<progress style="width:100%" max="100" value="{percent}"></progress>'
    value = '' if percent is None else f' · {percent:.0f}%'
    progress_display.update(HTML(f'<b>{labels[event["stage"]]}{value}</b><br>{meter}<br>{event["elapsed"]:.1f} seconds elapsed (including model loading)'))

workflow.run_generation(PYTHON, RUNNER, INPUT / "request.json", INPUT / "guide.wav", OUTPUT, show_progress)
metadata = json.loads((OUTPUT / "result.json").read_text())
print(f'Model generation: {metadata["generation_seconds"]:.1f} seconds. Quality still needs listening evaluation.')
display(Audio(filename=str(OUTPUT / "generated.wav")))
archive = JOB / ("result-" + REQUEST["instrument"] + "-" + JOB.name[:8] + ".zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for name in ("generated.wav", "result.json", "environment.json", "SUCCESS"):
        bundle.write(OUTPUT / name, arcname=name)
print("Downloading the result ZIP. After saving, unzip it and import generated.wav in AI Trial.")
DOWNLOAD = workflow.download_result(archive)
print(f'Result reached browser: {DOWNLOAD["bytes"]} bytes in {DOWNLOAD["seconds"]:.1f} seconds. Check your browser Downloads or save dialog.')


## Download again (only if needed)
If the browser blocked the download or it was interrupted, run this cell. It reuses the completed result and does not generate again. Transfer completion means bytes reached your browser; finish any save dialog there.


In [ ]:
if "archive" not in globals() or archive is None or not archive.is_file():
    raise RuntimeError("No completed result ZIP yet. Run Upload → Generate → Download first.")
DOWNLOAD = workflow.download_result(archive)
print(DOWNLOAD)


## Evaluate locally
Download before deleting the runtime. Unzip into an ignored local trial folder and run:
```
HumGenerationTrial transcribe --audio generated.wav --bpm 120 --instrument piano --output NEW_LOCAL_DIR
```
Use `tempo_bpm` and `instrument` from `result.json`. Compare the generated audio with `midi-reference.wav` (a generic tone representing the detected notes), and inspect `generated.mid` in a MIDI editor. Fill in the nine-row evaluation sheet; do not treat generated MIDI as verified transcription.

Repeat the Upload → Generate → Download cell for the other explicitly selected combinations. When finished, use **Runtime → Disconnect and delete runtime**. Keep the notebook private and clear outputs before sharing it.
